In [1]:
import gymnasium as gym
import numpy as np

from stable_baselines3 import PPO
from stable_baselines3.ppo.policies import MlpPolicy

In [2]:
env = gym.make("CartPole-v1")

model = PPO(MlpPolicy, env, verbose=0)

In [3]:
from stable_baselines3.common.base_class import BaseAlgorithm


def evaluate(
    model: BaseAlgorithm,
    num_episodes: int = 100,
    deterministic: bool = True,
) -> float:
    """
    Evaluate an RL agent for `num_episodes`.

    :param model: the RL Agent
    :param env: the gym Environment
    :param num_episodes: number of episodes to evaluate it
    :param deterministic: Whether to use deterministic or stochastic actions
    :return: Mean reward for the last `num_episodes`
    """
    # This function will only work for a single environment
    vec_env = model.get_env()
    obs = vec_env.reset()
    all_episode_rewards = []
    for _ in range(num_episodes):
        episode_rewards = []
        done = False
        # Note: SB3 VecEnv resets automatically:
        # https://stable-baselines3.readthedocs.io/en/master/guide/vec_envs.html#vecenv-api-vs-gym-api
        # obs = vec_env.reset()
        while not done:
            # _states are only useful when using LSTM policies
            # `deterministic` is to use deterministic actions
            action, _states = model.predict(obs, deterministic=deterministic)
            # here, action, rewards and dones are arrays
            # because we are using vectorized env
            obs, reward, done, _info = vec_env.step(action)
            episode_rewards.append(reward)

        all_episode_rewards.append(sum(episode_rewards))

    mean_episode_reward = np.mean(all_episode_rewards)
    print(f"Mean reward: {mean_episode_reward:.2f} - Num episodes: {num_episodes}")

    return mean_episode_reward

In [4]:
# Random Agent, before training
mean_reward_before_train = evaluate(model, num_episodes=100, deterministic=True)

Mean reward: 95.59 - Num episodes: 100


In [5]:
from stable_baselines3.common.evaluation import evaluate_policy

mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100, warn=False)

print(f"mean_reward: {mean_reward:.2f} +/- {std_reward:.2f}")

mean_reward: 97.70 +/- 24.95


In [6]:
# Train the agent for 10000 steps
model.learn(total_timesteps=10_000)

In [7]:
# Evaluate the trained agent
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=100)

print(f"mean_reward:{mean_reward:.2f} +/- {std_reward:.2f}")

/opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


mean_reward:386.42 +/- 115.64


In [8]:
# Set up fake display; otherwise rendering will fail
import os
os.system("Xvfb :1 -screen 0 1024x768x24 &")
os.environ['DISPLAY'] = ':1'

sh: Xvfb: command not found


In [9]:
import base64
from pathlib import Path

from IPython import display as ipythondisplay


def show_videos(video_path="", prefix=""):
    """
    Taken from https://github.com/eleurent/highway-env

    :param video_path: (str) Path to the folder containing videos
    :param prefix: (str) Filter the video, showing only the only starting with this prefix
    """
    html = []
    for mp4 in Path(video_path).glob("{}*.mp4".format(prefix)):
        video_b64 = base64.b64encode(mp4.read_bytes())
        html.append(
            """<video alt="{}" autoplay 
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{}" type="video/mp4" />
                </video>""".format(
                mp4, video_b64.decode("ascii")
            )
        )
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))

In [10]:
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv


def record_video(env_id, model, video_length=500, prefix="", video_folder="videos/"):
    """
    :param env_id: (str)
    :param model: (RL model)
    :param video_length: (int)
    :param prefix: (str)
    :param video_folder: (str)
    """
    eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])
    # Start the video at step=0 and record 500 steps
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda step: step == 0,
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = eval_env.reset()
    for _ in range(video_length):
        action, _ = model.predict(obs)
        obs, _, _, _ = eval_env.step(action)

    # Close the video recorder
    eval_env.close()

In [13]:
record_video("CartPole-v1", model, video_length=500, prefix="ppo-cartpole")

objc[42927]: Class SDL_RumbleMotor is implemented in both /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x30c6f8d40) and /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x31425c9c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[42927]: Class SDL_RumbleContext is implemented in both /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x30c6f8d90) and /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x31425ca18). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[42927]: Class SDLApplication is implemented in both /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x30c6f8890) and /opt/anaconda3/envs/ppo_lab/lib/python3.12/site-packages/pyg

Saving video to /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/videos/ppo-cartpole-step-0-to-step-500.mp4
MoviePy - Building video /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/videos/ppo-cartpole-step-0-to-step-500.mp4.
MoviePy - Writing video /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/videos/ppo-cartpole-step-0-to-step-500.mp4



MoviePy - Done !
MoviePy - video ready /Users/tanruoying/Desktop/ppo-lab/rl-tutorial-jnrr19/videos/ppo-cartpole-step-0-to-step-500.mp4


In [14]:
show_videos("videos", prefix="ppo")

In [15]:
model = PPO('MlpPolicy', "CartPole-v1", verbose=1).learn(1000)

Using cpu device
Creating environment from the given name 'CartPole-v1'
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22.4     |
|    ep_rew_mean     | 22.4     |
| time/              |          |
|    fps             | 9795     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
